# Gemma AST Agent — Colab runner

Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Results are saved to Google Drive, so a disconnect loses nothing: re-run and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive (results + the code zip live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/gemma-ast-agent-results'
import os; os.makedirs(RESULTS, exist_ok=True)

## 2. Get the code
Use **A** once the repo is on GitHub. Until then use **B**: upload `gemma-ast-agent.zip` to the top of your Drive (My Drive).

In [ ]:
REPO_URL = ''  # A: e.g. 'https://github.com/<you>/gemma-ast-agent.git'
ZIP_PATH = '/content/drive/MyDrive/gemma-ast-agent.zip'  # B

%cd /content
!rm -rf /content/gemma-ast-agent
if REPO_URL:
    !git clone -q $REPO_URL /content/gemma-ast-agent
else:
    !unzip -q -o "$ZIP_PATH" -d /content
%cd /content/gemma-ast-agent
!ls

## 3. Install and run the tests (no GPU needed; should say 15 passed)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes kagglehub pytest
!python -m pytest -q tests

## 4. Harness check with known fixes (every task should be RESOLVED)

In [ ]:
!python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend gold --conditions full --out /content/gold 2>&1 | tail -12

## 5. Download Gemma 4 from Kaggle
Open the Gemma 4 model page on Kaggle, accept the license, pick the **Transformers** variant of a small instruction-tuned model (E2B or E4B, `-it`), and copy its handle into `MODEL_HANDLE`.

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kagglehub
MODEL_HANDLE = ''  # paste from the Kaggle model page
MODEL_PATH = kagglehub.model_download(MODEL_HANDLE)
print(MODEL_PATH)
!ls $MODEL_PATH

## 6. Smoke test: 2 tasks, 1 condition
If you get a CUDA out-of-memory error, add `--load-in-4bit` (or use E2B).

In [ ]:
!python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend hf --model "$MODEL_PATH" \
  --dtype float16 --conditions ast_graph --limit 2 --max-new-tokens 1024 \
  --out "$RESULTS/smoke" 2>&1 | grep -v ' INFO '

Look at what the model actually wrote (first attempt of the first task):

In [ ]:
import json
r = json.loads(open(f'{RESULTS}/smoke/results.jsonl').readline())
print('resolved:', r['resolved'], '| apply errors:', r['attempts'][0]['apply_errors'])
print(r['attempts'][0]['response'][:3000])

## 7. Full grid on the sample benchmark
For the paper, replace `--tasks` with your generated benchmark from a real library (see README).

In [ ]:
!python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend hf --model "$MODEL_PATH" \
  --dtype float16 --conditions file,ast_graph,ast_graph_sc,full,oracle --max-new-tokens 1024 \
  --out "$RESULTS/sample" 2>&1 | grep -v ' INFO '

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{RESULTS}/sample/summary.md').read())